# Strict Candidate Retention Filter v1

Notebook-first evidence for final model retention. This run combines the strict walk-forward gate, SOXLBacktest-to-Qlib reference screen, and three-fold checkpoint selector into one conservative retention table. Only profiles that pass the strict walk-forward gate can retain an actor bundle; references and selectors remain diagnostic comparison evidence.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "SPEC.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "SPEC.md").exists():
    raise RuntimeError("Could not locate SOXL workspace root from notebook cwd")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.strict_candidate_retention_filter import (
    derive_strict_candidate_retention_filter,
    render_strict_candidate_retention_filter,
)

CARD = ROOT / "runtime/model_selection/card/strict_candidate_retention_filter_v1"
CARD.mkdir(parents=True, exist_ok=True)

STRICT_GATE_PATH = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
SOXLBACKTEST_PATH = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json"
CHECKPOINT_CARD = ROOT / "runtime/model_selection/card/cap10_three_fold_checkpoint_drift_selector_v1"
CHECKPOINT_SELECTION_PATH = CHECKPOINT_CARD / "selection.json"
CHECKPOINT_BY_FOLD_PATH = CHECKPOINT_CARD / "checkpoint_selection_by_fold.csv"
CHECKPOINT_METRICS_PATH = CHECKPOINT_CARD / "checkpoint_candidate_metrics.csv"

SOURCES = [
    ("strict_walkforward_gate", STRICT_GATE_PATH, "strict gate payload and current retention rows"),
    ("soxlbacktest_qlib_screen", SOXLBACKTEST_PATH, "SOXLBacktest converted-to-Qlib reference screen"),
    ("checkpoint_selector", CHECKPOINT_SELECTION_PATH, "three-fold checkpoint selector decision"),
    ("checkpoint_selection_by_fold", CHECKPOINT_BY_FOLD_PATH, "per-fold selected checkpoint gates"),
    ("checkpoint_candidate_metrics", CHECKPOINT_METRICS_PATH, "BC/PPO candidate drift and relative-return metrics"),
]

manifest = pd.DataFrame(
    [
        {
            "source_label": label,
            "path": str(path.relative_to(ROOT)),
            "exists": path.exists(),
            "role": role,
        }
        for label, path, role in SOURCES
    ]
)
manifest

,source_label,path,exists,role
0,strict_walkforward_gate,runtime/model_selection/card/model_retention_w...,True,strict gate payload and current retention rows
1,soxlbacktest_qlib_screen,runtime/model_selection/card/soxlbacktest_qlib...,True,SOXLBacktest converted-to-Qlib reference screen
2,checkpoint_selector,runtime/model_selection/card/cap10_three_fold_...,True,three-fold checkpoint selector decision
3,checkpoint_selection_by_fold,runtime/model_selection/card/cap10_three_fold_...,True,per-fold selected checkpoint gates
4,checkpoint_candidate_metrics,runtime/model_selection/card/cap10_three_fold_...,True,BC/PPO candidate drift and relative-return met...


In [2]:
missing = manifest.loc[~manifest["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required sources: {missing}")

strict_gate_payload = json.loads(STRICT_GATE_PATH.read_text())
soxlbacktest_summary = json.loads(SOXLBACKTEST_PATH.read_text())
checkpoint_selection = json.loads(CHECKPOINT_SELECTION_PATH.read_text())
checkpoint_selection_by_fold = pd.read_csv(CHECKPOINT_BY_FOLD_PATH)
checkpoint_candidate_metrics = pd.read_csv(CHECKPOINT_METRICS_PATH)

result = derive_strict_candidate_retention_filter(
    strict_gate_payload=strict_gate_payload,
    soxlbacktest_summary=soxlbacktest_summary,
    checkpoint_selection=checkpoint_selection,
    checkpoint_selection_by_fold=checkpoint_selection_by_fold,
    checkpoint_candidate_metrics=checkpoint_candidate_metrics,
)

rows = pd.DataFrame(result["rows"])
summary = result["summary"]
rows[[
    "candidate_id",
    "candidate_type",
    "promotion_allowed",
    "actor_retention_allowed",
    "diagnostic_visible_allowed",
    "primary_blocker",
    "required_next_mechanism",
]].sort_values(["candidate_type", "candidate_id"])

,candidate_id,candidate_type,promotion_allowed,actor_retention_allowed,diagnostic_visible_allowed,primary_blocker,required_next_mechanism
14,cap10_three_fold_checkpoint_drift_selector_v1,checkpoint_selector,False,False,True,trend_underparticipation,train_bull_trend_participation_with_drawdown_d...
1,balanced,profile,False,False,True,missing_wf_window:wf_2020,complete_walkforward_evaluation
2,balanced_macro_v1,profile,False,False,True,missing_wf_window:wf_2020,complete_walkforward_evaluation
6,balanced_rule_residual_imitation_v4,profile,False,False,True,wf_max_drawdown_ge_20:wf_2020,strict_walkforward_training_objective
3,balanced_rule_residual_v19,profile,False,False,True,wf_max_drawdown_ge_20:wf_2020,strict_walkforward_training_objective
10,balanced_rule_residual_v19_reg,profile,False,False,True,wf_max_drawdown_ge_20:wf_2022,strict_walkforward_training_objective
11,balanced_rule_residual_v19_reg_light,profile,False,False,True,wf_max_drawdown_ge_20:wf_2020,strict_walkforward_training_objective
12,balanced_rule_residual_v19_reg_v22_retest_v1,profile,False,False,True,missing_wf_window:wf_2020,complete_walkforward_evaluation
5,balanced_rule_residual_v22,profile,False,False,True,wf_max_drawdown_ge_20:wf_2020,strict_walkforward_training_objective
7,drawdown_rebound_reward_train_v1,profile,False,False,True,wf_max_drawdown_ge_20:wf_2020,strict_walkforward_training_objective


In [3]:
sidecars = {
    "candidate_retention_filter_csv": CARD / "candidate_retention_filter.csv",
    "selection_json": CARD / "selection.json",
    "summary_json": CARD / "summary.json",
    "markdown": CARD / "strict_candidate_retention_filter.md",
    "source_manifest_csv": CARD / "source_manifest.csv",
}

rows.to_csv(sidecars["candidate_retention_filter_csv"], index=False)
manifest.to_csv(sidecars["source_manifest_csv"], index=False)
sidecars["markdown"].write_text(render_strict_candidate_retention_filter(result))

selection = {
    "experiment": "strict_candidate_retention_filter_v1",
    "decision": summary["decision"],
    "promotion_allowed": summary["promotable_profile_count"] > 0,
    "actor_retention_allowed": summary["actor_retention_allowed_count"] > 0,
    "ppo_allowed_next": summary["ppo_allowed_next"],
    "reference_baseline": summary["reference_baseline"],
    "source_paths": manifest[["source_label", "path", "role"]].to_dict(orient="records"),
    "sidecars": {key: str(path.relative_to(ROOT)) for key, path in sidecars.items()},
}
sidecars["selection_json"].write_text(json.dumps(selection, indent=2, sort_keys=True))

summary_payload = {
    "experiment": "strict_candidate_retention_filter_v1",
    "summary": summary,
    "row_count": int(len(rows)),
    "candidate_type_counts": rows["candidate_type"].value_counts().sort_index().to_dict(),
    "source_manifest": manifest.to_dict(orient="records"),
    "sidecars": {key: str(path.relative_to(ROOT)) for key, path in sidecars.items()},
}
sidecars["summary_json"].write_text(json.dumps(summary_payload, indent=2, sort_keys=True))

print(json.dumps(summary, indent=2, sort_keys=True))

{
  "actor_retention_allowed_count": 0,
  "decision": "strict_candidate_retention_filter_no_promotable_candidates",
  "max_walkforward_drawdown_pct": 20.0,
  "next_mechanism": "train_bull_trend_participation_with_drawdown_drift_stop_no_promotion",
  "passing_profile_count": 0,
  "ppo_allowed_next": false,
  "profiles_scanned": 12,
  "promotable_profile_count": 0,
  "reference_baseline": "soxl_backtest_policy_reimplementation_v1",
  "reference_visible_count": 2,
  "required_best_return_pct": 213.143462,
  "required_wf_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ]
}


## Conclusion

This run produced `candidate_retention_filter.csv`, `selection.json`, `summary.json`, `strict_candidate_retention_filter.md`, and `source_manifest.csv` under `runtime/model_selection/card/strict_candidate_retention_filter_v1/`.

Decision: `strict_candidate_retention_filter_no_promotable_candidates`. The strict gate currently has zero promotable trained profiles, so existing trained actors remain diagnostic-only until a profile passes every required walk-forward window and the return-alignment gate.

SOXLBacktest-to-Qlib remains visible as a reference screen, not a runnable actor and not a final retained model. The three-fold checkpoint selector is also diagnostic-only: `wf_2020` is the failed fold, the primary blocker is trend underparticipation, and PPO continuation is stopped because the PPO drift path exceeds the drawdown-drift guard.

Next action: train or rescore the bull-trend participation objective with the drawdown-drift stop, then rerun the strict walk-forward retention gate before allowing any actor checkpoint to become the final retained model.